1- **Create Table on BigQuery**

In [1]:
from google.cloud import bigquery
from google.oauth2 import service_account

# =========================================
# AUTHENTICATION
# =========================================

SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE
)

PROJECT_ID = "depihealthnux"
DATASET_ID = "Depihealthnux_Main"
TABLE_ID = "Booked_Visits"

TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

client = bigquery.Client(
    credentials=credentials,
    project=PROJECT_ID
)

# =========================================
# SCHEMA
# =========================================

schema = [

    bigquery.SchemaField("Booking_Key","STRING",mode="REQUIRED"),

    bigquery.SchemaField("Visit_Key","STRING"),

    bigquery.SchemaField("Dr_Code","STRING"),
    bigquery.SchemaField("Speciality","STRING"),
    bigquery.SchemaField("Dr_Name","STRING"),

    bigquery.SchemaField("Patient_U_ID","STRING"),

    bigquery.SchemaField("Scheduled_Day","STRING"),

    bigquery.SchemaField("Scheduled_Date","DATE"),

    bigquery.SchemaField("Scheduled_Start_Time","TIME"),
    bigquery.SchemaField("Scheduled_End_Time","TIME"),

    bigquery.SchemaField("Shift_Name","STRING"),

    bigquery.SchemaField("Status","STRING"),

    bigquery.SchemaField("Creation_Time_Stamp","TIMESTAMP"),

    bigquery.SchemaField("Creation_By","STRING"),

    bigquery.SchemaField("Chief_Complaint","STRING"),

    bigquery.SchemaField("Diagnosis_Codes","STRING"),

    bigquery.SchemaField("Requested_Labs","STRING"),

    bigquery.SchemaField("Requested_Scans","STRING"),

    bigquery.SchemaField("Doctor_Notes","STRING"),

    bigquery.SchemaField("Consultation_Timestamp","TIMESTAMP")

]

table = bigquery.Table(
    TABLE_REF,
    schema=schema
)

# =========================================
# PARTITIONING
# =========================================

table.time_partitioning = bigquery.TimePartitioning(
    type_=bigquery.TimePartitioningType.DAY,
    field="Scheduled_Date"
)

# =========================================
# CLUSTERING
# =========================================

table.clustering_fields = [
    "Dr_Code",
    "Patient_U_ID",
    "Status"
]

client.create_table(
    table,
    exists_ok=True
)

print("=================================")
print("BigQuery table created successfully")
print(TABLE_REF)
print("=================================")

BigQuery table created successfully
depihealthnux.Depihealthnux_Main.Booked_Visits


2- **Create Table on Postgres**

In [2]:
import sys
import psycopg2

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# CONNECT
# =========================================

conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

# =========================================
# CREATE SEQUENCE
# =========================================

cursor.execute("""

CREATE SEQUENCE IF NOT EXISTS booking_no_seq;

""")

# =========================================
# CREATE TABLE
# =========================================

create_table_query = """
CREATE TABLE IF NOT EXISTS Booked_Visits (

    Booking_Key TEXT PRIMARY KEY
    DEFAULT (
        'B' ||
        LPAD(
            nextval('booking_no_seq')::TEXT,
            6,
            '0'
        )
    ),

    Visit_Key TEXT NOT NULL,

    Patient_U_ID TEXT NOT NULL,

    Status TEXT,

    Creation_Time_Stamp TIMESTAMP
    DEFAULT CURRENT_TIMESTAMP,

    Creation_By TEXT,

    Chief_Complaint TEXT,

    Diagnosis_Codes TEXT,

    Requested_Labs TEXT,

    Requested_Scans TEXT,

    Doctor_Notes TEXT,

    Consultation_Timestamp TIMESTAMP

);
"""

cursor.execute(create_table_query)

# =========================================
# INDEXES
# =========================================

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_booking_visit
ON Booked_Visits(Visit_Key);

""")

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_booking_patient
ON Booked_Visits(Patient_U_ID);

""")

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_booking_status
ON Booked_Visits(Status);

""")

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_booking_created
ON Booked_Visits(Creation_Time_Stamp);

""")

conn.commit()

print("=================================")
print("PostgreSQL table created successfully")
print("Table: Booked_Visits")
print("=================================")

cursor.close()
conn.close()

PostgreSQL table created successfully
Table: Booked_Visits


**Sync BigQuery to Postgres**

In [4]:
import sys
import pandas as pd
import psycopg2

from psycopg2.extras import execute_values

from google.cloud import bigquery
from google.oauth2 import service_account

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

credentials = service_account.Credentials.from_service_account_file(
    "../Keys/BigQueryKey.json"
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

query = """

SELECT

    Booking_Key,
    Visit_Key,
    Patient_U_ID,
    Status,
    Creation_Time_Stamp,
    Creation_By,
    Chief_Complaint,
    Diagnosis_Codes,
    Requested_Labs,
    Requested_Scans,
    Doctor_Notes,
    Consultation_Timestamp

FROM `depihealthnux.Depihealthnux_Main.Booked_Visits`

ORDER BY Booking_Key

"""

df = client.query(query).to_dataframe()

print(df.head(3))
print(f"Rows Retrieved: {len(df)}")

conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

cursor.execute("""
DELETE FROM Booked_Visits;
""")

if not df.empty:

    execute_values(
        cursor,
        """
        INSERT INTO Booked_Visits (

            Booking_Key,
            Visit_Key,
            Patient_U_ID,
            Status,
            Creation_Time_Stamp,
            Creation_By,
            Chief_Complaint,
            Diagnosis_Codes,
            Requested_Labs,
            Requested_Scans,
            Doctor_Notes,
            Consultation_Timestamp

        )
        VALUES %s
        """,
        df.values.tolist(),
        page_size=1000
    )

cursor.execute("""

SELECT setval(
    'booking_no_seq',
    COALESCE(
        (
            SELECT MAX(
                CAST(
                    REPLACE(Booking_Key,'B','')
                    AS INTEGER
                )
            )
            FROM Booked_Visits
        ),
        1
    ),
    true
);

""")

conn.commit()

print(f"Inserted {len(df)} rows")

cursor.execute("""

SELECT *
FROM Booked_Visits
ORDER BY Booking_Key
LIMIT 3

""")

print("\nFirst 3 Rows From PostgreSQL")

for row in cursor.fetchall():
    print(row)

cursor.close()
conn.close()

Empty DataFrame
Columns: [Booking_Key, Visit_Key, Patient_U_ID, Status, Creation_Time_Stamp, Creation_By, Chief_Complaint, Diagnosis_Codes, Requested_Labs, Requested_Scans, Doctor_Notes, Consultation_Timestamp]
Index: []
Rows Retrieved: 0
Inserted 0 rows

First 3 Rows From PostgreSQL


**Sync Postgres to BigQuery**

In [18]:
import sys
import pandas as pd
import psycopg2

from google.cloud import bigquery
from google.oauth2 import service_account

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

credentials = service_account.Credentials.from_service_account_file(
    "../Keys/BigQueryKey.json"
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

conn = psycopg2.connect(POSTGRES_URL)

query = """

SELECT

    b.Booking_Key,

    b.Visit_Key,

    tt.Dr_Code,

    d.Speciality,

    d.Dr_Name,

    b.Patient_U_ID,

    tt.Day_of_Week AS Scheduled_Day,

    av.Scheduled_Date,

    tt.Scheduled_Start_Time,

    tt.Scheduled_End_Time,

    CONCAT(
        d.Dr_Name,
        ' - ',
        d.Speciality,
        ' - ',
        tt.Day_of_Week,
        ' - ',
        TO_CHAR(av.Scheduled_Date,'YYYY-MM-DD'),
        ' - ',
        TO_CHAR(tt.Scheduled_Start_Time,'HH12:MI AM')
    ) AS Shift_Name,

    b.Status,

    b.Creation_Time_Stamp,

    b.Creation_By,

    b.Chief_Complaint,

    b.Diagnosis_Codes,

    b.Requested_Labs,

    b.Requested_Scans,

    b.Doctor_Notes,

    b.Consultation_Timestamp

FROM Booked_Visits b

LEFT JOIN Available_Visits av
    ON b.Visit_Key = av.Visit_Key

LEFT JOIN Dr_Time_Table tt
    ON av.Time_Table_Key = tt.Time_Table_Key

LEFT JOIN Dr_List d
    ON tt.Dr_Code = d.Dr_Code

ORDER BY b.Booking_Key

"""

df = pd.read_sql(query, conn)

conn.close()

# =========================================
# FIX DATA TYPES FOR BIGQUERY
# =========================================

# DATE Partition Column
df["scheduled_date"] = pd.to_datetime(
    df["scheduled_date"]
).dt.date

# TIME Columns
df["scheduled_start_time"] = pd.to_datetime(
    df["scheduled_start_time"].astype(str),
    format="%H:%M:%S"
).dt.time

df["scheduled_end_time"] = pd.to_datetime(
    df["scheduled_end_time"].astype(str),
    format="%H:%M:%S"
).dt.time

# =========================================
# VERIFY DATA TYPES
# =========================================

print("=================================")
print("Data Types")
print("=================================")
print(df.dtypes)

print("\n=================================")
print("First 3 Rows From PostgreSQL")
print("=================================")
print(df.head(3))

# =========================================
# LOAD TO BIGQUERY
# =========================================
print(df.columns.tolist())
print(df.dtypes)
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)
if df.empty:

    print("=================================")
    print("No Booked Visits Found")
    print("BigQuery Sync Skipped")
    print("=================================")

else:

    job = client.load_table_from_dataframe(
        df,
        "depihealthnux.Depihealthnux_Main.Booked_Visits",
        job_config=job_config
    )

    job.result()

    print(f"Uploaded {len(df)} rows")

# =========================================
# VERIFY BIGQUERY DATA
# =========================================

verify_df = client.query("""

SELECT *
FROM `depihealthnux.Depihealthnux_Main.Booked_Visits`
LIMIT 3

""").to_dataframe()

print("\nFirst 3 Rows From BigQuery")

print(verify_df)

C:\Users\Sedeek Elmasry\AppData\Local\Temp\ipykernel_15280\2624544019.py:91: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Data Types
booking_key                       object
visit_key                         object
dr_code                           object
speciality                        object
dr_name                           object
patient_u_id                      object
scheduled_day                     object
scheduled_date                    object
scheduled_start_time              object
scheduled_end_time                object
shift_name                        object
status                            object
creation_time_stamp       datetime64[ns]
creation_by                       object
chief_complaint                   object
diagnosis_codes                   object
requested_labs                    object
requested_scans                   object
doctor_notes                      object
consultation_timestamp    datetime64[ns]
dtype: object

First 3 Rows From PostgreSQL
  booking_key visit_key dr_code speciality    dr_name patient_u_id  \
0     B000001  VK000001   Dr001      Chest  Dr. Wafaa  

**Sample Data**

In [1]:
import sys
import psycopg2

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# CONNECT TO POSTGRES
# =========================================

conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

# =========================================
# INSERT INTO DR_TIME_TABLE
# =========================================

cursor.execute("""

INSERT INTO Dr_Time_Table (

    Time_Table_Key,
    Dr_Code,
    Day_of_Week,
    Scheduled_Start_Time,
    Scheduled_End_Time,
    Capacity,
    Is_Active

)

VALUES

('TT001','Dr001','Monday','18:00:00','21:00:00',6,'TRUE'),

('TT002','Dr001','Tuesday','10:00:00','12:00:00',7,'TRUE'),

('TT003','Dr002','Saturday','10:00:00','12:00:00',8,'TRUE')

ON CONFLICT (Time_Table_Key) DO NOTHING;

""")

# =========================================
# INSERT INTO AVAILABLE_VISITS
# =========================================

cursor.execute("""

INSERT INTO Available_Visits (

    Visit_Key,
    Time_Table_Key,
    Scheduled_Date,
    Capacity,
    Remaining_Capacity

)

VALUES

('VK000001','TT001','2026-06-01',6,3),
('VK000002','TT001','2026-06-08',6,6),
('VK000003','TT001','2026-06-15',6,6),
('VK000004','TT001','2026-06-22',6,6),

('VK000005','TT003','2026-06-06',8,5),
('VK000006','TT003','2026-06-13',8,8),
('VK000007','TT003','2026-06-20',8,8),
('VK000008','TT003','2026-06-27',8,8)

ON CONFLICT (Visit_Key) DO NOTHING;

""")

# =========================================
# INSERT INTO BOOKED_VISITS
# =========================================

cursor.execute("""

INSERT INTO Booked_Visits (

    Booking_Key,
    Visit_Key,
    Patient_U_ID,
    Status,
    Creation_Time_Stamp,
    Creation_By,
    Chief_Complaint,
    Diagnosis_Codes,
    Requested_Labs,
    Requested_Scans,
    Doctor_Notes,
    Consultation_Timestamp

)

VALUES

(
'B000001',
'VK000001',
'PAT_000001',
'Done',
'2026-05-31 22:05:00',
'Admin',
'geh t3ban bla bla',
'["E10"]',
NULL,
'["SC1"]',
NULL,
'2026-06-01 22:00:00'
),

(
'B000002',
'VK000001',
'PAT_000003',
'Done',
'2026-05-31 22:10:00',
'Admin',
'geh t3ban bla bla',
'["E10","E55"]',
'["LC1","LC9"]',
'["SC2","SC7"]',
'Mfesh',
'2026-06-01 23:00:00'
),

(
'B000003',
'VK000001',
'PAT_000002',
'Done',
'2026-05-31 23:00:00',
'Admin',
'geh t3ban bla bla',
'["E55"]',
'["LC10"]',
NULL,
NULL,
'2026-06-02 00:00:00'
),

(
'B000004',
'VK000005',
'PAT_000002',
'Scheduled',
'2026-06-04 23:00:00',
'Admin',
NULL,
NULL,
NULL,
NULL,
NULL,
NULL
),

(
'B000005',
'VK000005',
'PAT_000001',
'Scheduled',
'2026-06-04 23:05:00',
'Admin',
NULL,
NULL,
NULL,
NULL,
NULL,
NULL
),

(
'B000006',
'VK000005',
'PAT_000004',
'Scheduled',
'2026-06-04 23:30:00',
'Admin',
NULL,
NULL,
NULL,
NULL,
NULL,
NULL
)

ON CONFLICT (Booking_Key) DO NOTHING;

""")

# =========================================
# COMMIT
# =========================================

conn.commit()

print("=================================")
print("Test Data Inserted Successfully")
print("=================================")

# =========================================
# VERIFY FULL JOIN
# =========================================

cursor.execute("""

SELECT

    b.Booking_Key,

    b.Visit_Key,

    tt.Dr_Code,

    d.Speciality,

    d.Dr_Name,

    b.Patient_U_ID,

    tt.Day_of_Week,

    av.Scheduled_Date,

    tt.Scheduled_Start_Time,

    tt.Scheduled_End_Time,

    b.Status

FROM Booked_Visits b

LEFT JOIN Available_Visits av
    ON b.Visit_Key = av.Visit_Key

LEFT JOIN Dr_Time_Table tt
    ON av.Time_Table_Key = tt.Time_Table_Key

LEFT JOIN Dr_List d
    ON tt.Dr_Code = d.Dr_Code

ORDER BY b.Booking_Key

LIMIT 10

""")

print("\n=================================")
print("Join Verification")
print("=================================")

for row in cursor.fetchall():
    print(row)

# =========================================
# COUNTS
# =========================================

cursor.execute("SELECT COUNT(*) FROM Dr_Time_Table")
print(f"\nDr_Time_Table Rows: {cursor.fetchone()[0]}")

cursor.execute("SELECT COUNT(*) FROM Available_Visits")
print(f"Available_Visits Rows: {cursor.fetchone()[0]}")

cursor.execute("SELECT COUNT(*) FROM Booked_Visits")
print(f"Booked_Visits Rows: {cursor.fetchone()[0]}")

# =========================================
# CLOSE
# =========================================

cursor.close()
conn.close()

Test Data Inserted Successfully

Join Verification
('B000001', 'VK000001', 'Dr001', 'Chest', 'Dr. Wafaa', 'PAT_000001', 'Monday', datetime.date(2026, 6, 1), datetime.time(18, 0), datetime.time(21, 0), 'Done')
('B000002', 'VK000001', 'Dr001', 'Chest', 'Dr. Wafaa', 'PAT_000003', 'Monday', datetime.date(2026, 6, 1), datetime.time(18, 0), datetime.time(21, 0), 'Done')
('B000003', 'VK000001', 'Dr001', 'Chest', 'Dr. Wafaa', 'PAT_000002', 'Monday', datetime.date(2026, 6, 1), datetime.time(18, 0), datetime.time(21, 0), 'Done')
('B000004', 'VK000005', 'Dr002', 'Gastroenterology', 'Dr. Angie', 'PAT_000002', 'Saturday', datetime.date(2026, 6, 6), datetime.time(10, 0), datetime.time(12, 0), 'Scheduled')
('B000005', 'VK000005', 'Dr002', 'Gastroenterology', 'Dr. Angie', 'PAT_000001', 'Saturday', datetime.date(2026, 6, 6), datetime.time(10, 0), datetime.time(12, 0), 'Scheduled')
('B000006', 'VK000005', 'Dr002', 'Gastroenterology', 'Dr. Angie', 'PAT_000004', 'Saturday', datetime.date(2026, 6, 6), date